# Court LLM Descriptor Extractor — Google Colab GPU

This notebook runs the expensive LLM descriptor extraction step in **Google Colab** and writes raw minimal semantic descriptors to Google Drive.

Target runtime used for this version:

```text
Running in Colab: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, ~96 GiB VRAM
Model: Qwen/Qwen3-8B-AWQ
```

Expected project layout:

```text
/content/drive/MyDrive/swiss_law/
├── data/
│   └── court_considerations.csv
├── models/              # optional Hugging Face/vLLM download cache
└── outputs/             # generated JSONL/CSV/metrics
```

This notebook is intentionally **not** the final enrichment pipeline.

It writes:

```text
court_considerations.csv
→ Qwen3-8B-AWQ
→ outputs/court_llm_descriptors_*.jsonl
```

Do **not** do anchor normalization here. Do **not** build final retrieval views here.

After downloading or syncing the output JSONL, run the local CPU scripts:

```text
scripts/court_enrichment_profile.py
scripts/court_enrichment_normalizer.py
scripts/finalize_court_enrichment_from_llm.py
```

The local finalizer will merge the raw LLM descriptors with `court_considerations.csv`, normalize anchors, correct role/outcome, build retrieval views, and produce the final production JSONL.


In [1]:
# Cell 0 — Colab setup and package install

import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# vLLM is the primary runtime for AWQ inference here.
# After the first install in a fresh Colab runtime, restart the runtime once if imports fail.
!pip install -q -U "vllm>=0.8.5" "transformers>=4.51.0" accelerate safetensors pandas tqdm huggingface_hub

print('Setup done. If this was the first package install in a fresh Colab runtime, restart runtime once, then continue from Cell 1.')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.20.0 requires flashinfer-python==0.6.8.post1, which is not installed.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-proto 1.41.1 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
google-cloud-videointelligence 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-vision 3.12.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
grpc-google-iam-v1 0.14.3 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but y

In [2]:
# Cell 1 — Imports and runtime check

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Optional
from collections import Counter
import os
import sys
import re
import gc
import ast
import json
import time
import traceback

import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
except Exception:
    torch = None

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = Path('/content/drive/MyDrive/swiss_law') if IN_COLAB else Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'

print('Imports OK')
print('Running in Colab:', IN_COLAB)
print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)

if torch is not None and torch.cuda.is_available():
    print('CUDA devices:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f'GPU {i}: {props.name}; free={free/1024**3:.2f} GiB / total={total/1024**3:.2f} GiB')
else:
    print('WARNING: CUDA GPU is not available. In Colab, select Runtime > Change runtime type > GPU.')


Imports OK
CUDA devices: 2
GPU 0: Tesla T4; free=14.46 GiB / total=14.56 GiB
GPU 1: Tesla T4; free=14.46 GiB / total=14.56 GiB


In [3]:
# Cell 2 — Config

@dataclass
class Config:
    # Colab / Drive paths
    base_dir: str = str(BASE_DIR)
    data_dir: str = str(DATA_DIR)
    output_dir: str = str(OUTPUT_DIR)
    model_download_dir: str = str(MODEL_DIR / 'huggingface')

    # Input CSV. Put court_considerations.csv in /content/drive/MyDrive/swiss_law/data/
    input_csv: str = str(DATA_DIR / 'court_considerations.csv')
    fallback_input_csv: str = 'court_considerations.csv'

    # Hugging Face AWQ model id
    model_name: str = 'Qwen/Qwen3-8B-AWQ'

    # Row selection
    start: int = 0
    limit: int = 50
    sample_random: bool = False
    random_seed: Optional[int] = 42
    min_text_chars: int = 80
    max_text_chars: int = 3200

    # GPU/vLLM. Single-GPU is enough for the RTX PRO 6000 Blackwell Server Edition runtime shown by Colab.
    gpu_mode: str = 'single'   # 'single' or 'tp2'
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.90
    max_model_len: int = 4096
    max_num_seqs: int = 16
    batch_size: int = 8
    enforce_eager: bool = False
    quantization: str = 'awq'
    disable_custom_all_reduce: bool = True
    force_triton_attention: bool = False

    # Structured output is disabled because some vLLM versions can crash with guided/structured decoding.
    use_structured_outputs: bool = False

    # Generation
    max_new_tokens: int = 384
    retry_max_new_tokens: int = 512
    max_retries: int = 1
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.02
    enable_thinking: bool = False

    # Output
    include_raw_output_on_success: bool = False

cfg = Config()

# Apply GPU mode before loading vLLM.
if cfg.gpu_mode == 'single':
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
    cfg.tensor_parallel_size = 1
elif cfg.gpu_mode == 'tp2':
    os.environ.pop('CUDA_VISIBLE_DEVICES', None)
    cfg.tensor_parallel_size = 2
    cfg.disable_custom_all_reduce = True
else:
    raise ValueError("cfg.gpu_mode must be 'single' or 'tp2'")

if cfg.force_triton_attention:
    os.environ.setdefault('VLLM_ATTENTION_BACKEND', 'TRITON_ATTN')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

base_dir = Path(cfg.base_dir)
data_dir = Path(cfg.data_dir)
out_dir = Path(cfg.output_dir)
model_download_dir = Path(cfg.model_download_dir)
for d in [base_dir, data_dir, out_dir, model_download_dir]:
    d.mkdir(parents=True, exist_ok=True)

end_idx = cfg.start + cfg.limit - 1 if cfg.limit else -1
suffix = f'{cfg.start:07d}_{end_idx:07d}' if cfg.limit else f'{cfg.start:07d}_all'
output_jsonl = out_dir / f'court_llm_descriptors_{suffix}.jsonl'
output_preview_csv = out_dir / f'court_llm_descriptors_{suffix}_preview.csv'
output_failures_jsonl = out_dir / f'court_llm_descriptors_{suffix}_failures.jsonl'
output_metrics_json = out_dir / f'court_llm_descriptors_{suffix}_metrics.json'

print(json.dumps(asdict(cfg), indent=2))
print('Output JSONL:', output_jsonl)


{'input_csv': '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv', 'fallback_input_csv': 'court_considerations.csv', 'model_name': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1', 'start': 0, 'limit': 50, 'sample_random': False, 'random_seed': 42, 'min_text_chars': 80, 'max_text_chars': 3200, 'gpu_mode': 'single', 'tensor_parallel_size': 1, 'gpu_memory_utilization': 0.78, 'max_model_len': 4096, 'max_num_seqs': 8, 'batch_size': 4, 'enforce_eager': True, 'quantization': 'awq_marlin', 'disable_custom_all_reduce': True, 'force_triton_attention': True, 'use_structured_outputs': False, 'max_new_tokens': 384, 'retry_max_new_tokens': 512, 'max_retries': 1, 'temperature': 0.0, 'top_p': 1.0, 'repetition_penalty': 1.02, 'enable_thinking': False, 'output_dir': '/kaggle/working', 'include_raw_output_on_success': False}
Output JSONL: /kaggle/working/court_llm_descriptors_0000000_0000049.jsonl


In [4]:
# Cell 3 — Load CSV and select rows

def resolve_input_path() -> Path:
    candidates = [
        Path(cfg.input_csv),
        DATA_DIR / cfg.fallback_input_csv,
        BASE_DIR / cfg.fallback_input_csv,
        Path('/content') / cfg.fallback_input_csv,
        Path.cwd() / cfg.fallback_input_csv,
    ]
    for p in candidates:
        if p.exists():
            return p

    # Helpful fallback search for Colab/Drive and local runs.
    search_roots = [DATA_DIR, BASE_DIR, Path('/content')]
    for root in search_roots:
        if root.exists():
            for pat in ['**/court_considerations.csv', '**/court_consideration.csv']:
                found = sorted(root.glob(pat))
                if found:
                    return found[0]

    raise FileNotFoundError(
        'Could not find court_considerations.csv. Expected location: '
        f'{DATA_DIR / cfg.fallback_input_csv}'
    )

input_path = resolve_input_path()
print('Using input:', input_path)

df = pd.read_csv(input_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))

citation_col = 'citation' if 'citation' in df.columns else df.columns[0]
text_col = 'text' if 'text' in df.columns else df.columns[1]

valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
valid[text_col] = valid[text_col].astype(str)
valid['_text_len'] = valid[text_col].str.strip().str.len()
valid = valid[valid['_text_len'] >= cfg.min_text_chars].copy()

if cfg.sample_random:
    pool = valid.iloc[cfg.start:] if cfg.start else valid
    work_df = pool.sample(n=min(cfg.limit, len(pool)), random_state=cfg.random_seed)
else:
    end = None if not cfg.limit else cfg.start + cfg.limit
    work_df = valid.iloc[cfg.start:end]

work_df = work_df.reset_index(drop=False).rename(columns={'index': '_source_row'})
print('Selected rows:', len(work_df))
display(work_df[['_source_row', citation_col, text_col, '_text_len']].head(20))


Using input: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv
Shape: (2476315, 2)
Columns: ['citation', 'text']
Selected rows: 50


,_source_row,citation,text,_text_len
0,1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...,885
1,2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents...",437
2,3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...,242
3,4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...,286
4,5,BGE 139 I 2 E. 7.1,S. 144) bestätigte das Verwaltungsgericht den ...,1493
5,6,BGE 139 I 2 E. 5.4,Strittig ist hier hingegen die Umsetzung der P...,183
6,7,BGE 139 I 2 E. 7.1,S. 70) dargestellten und im angefochtenen Ents...,690
7,8,BGE 139 I 2 E. 5.5,"Zu beachten ist sodann, dass nach der schwyzer...",904
8,9,BGE 139 I 2 E. 5.6,Die Umsetzung einer Planungsinitiative ist ver...,2405
9,10,BGE 139 I 2 E. 5.7,Die an der Volksabstimmung vom 26. November 20...,278


In [5]:
# Cell 4 — Minimal LLM descriptor schema and prompt

# The LLM intentionally does NOT produce anchors, outcomes, summaries, questions, or retrieval views.
# Local CPU scripts will build those deterministically later.

DESCRIPTOR_KEYS = [
    'legal_area',
    'primary_domain',
    'secondary_domain',
    'legal_domain_path',
    'topic',
    'subtopic',
    'micro_topic',
    'concepts_en',
    'terms_original',
    'doctrinal_rule',
    'legal_test',
    'fact_pattern_tags',
    'procedural_context',
    'paragraph_role',
    'authority_role',
    'specificity_score',
]

ROLE_VALUES = {
    'holding', 'reasoning', 'facts', 'procedural_history', 'legal_standard',
    'application', 'citation', 'costs', 'notification', 'disposition', 'neutral'
}

LLM_SCHEMA_HINT = {
    'legal_area': 'broad English legal area, max 6 words',
    'primary_domain': 'stable domain, max 6 words',
    'secondary_domain': 'narrow domain, max 8 words',
    'legal_domain_path': ['2-5 short taxonomy labels, broad to narrow'],
    'topic': 'specific topic, max 8 words',
    'subtopic': 'more specific subtopic, max 10 words',
    'micro_topic': 'most specific legal issue, max 14 words',
    'concepts_en': ['3-6 precise English legal concepts'],
    'terms_original': ['3-8 exact important source-language legal terms'],
    'doctrinal_rule': 'only if paragraph states a rule; otherwise empty; max 25 words',
    'legal_test': 'only if paragraph states/applies a test; otherwise empty; max 20 words',
    'fact_pattern_tags': ['0-5 concrete factual/procedural tags'],
    'procedural_context': 'short procedural posture if clear',
    'paragraph_role': 'holding|reasoning|facts|procedural_history|legal_standard|application|citation|costs|notification|disposition|neutral',
    'authority_role': ['0-3 legal value labels, e.g. legal_test, application_of_rule, background'],
    'specificity_score': 'number 0 to 1',
}

SYSTEM_PROMPT = '''You are a Swiss legal descriptor extractor.

Return exactly one compact JSON object.
Do not generate user questions.
Do not generate summaries.
Do not extract statute anchors.
Do not extract case anchors.
Do not infer final court outcome.
Do not build retrieval views.

Only extract query-neutral semantic descriptors that cannot be reliably obtained by static regex parsing.
Use English for classification fields.
Use exact German/French/Italian terms for terms_original.
If the paragraph is factual/procedural/boilerplate, keep doctrinal_rule and legal_test empty.
Use only information grounded in the citation text.
JSON only.'''

USER_TEMPLATE = '''Citation: {citation}

Text:
{text}

Required JSON shape:
{schema}

Return JSON only.'''

def trim_text(text: str, max_chars: int) -> str:
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head
    return text[:head].rstrip() + ' ... [TRUNCATED] ... ' + text[-tail:].lstrip()

def build_user_prompt(citation: str, text: str) -> str:
    return USER_TEMPLATE.format(
        citation=str(citation),
        text=trim_text(text, cfg.max_text_chars),
        schema=json.dumps(LLM_SCHEMA_HINT, ensure_ascii=False),
    )


In [6]:
# Cell 5 — JSON parsing and descriptor normalization

def extract_json_object(raw: str) -> dict[str, Any]:
    if raw is None:
        raise ValueError('empty model output')
    s = str(raw).strip()
    s = re.sub(r'^\s*```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```\s*$', '', s)
    s = re.sub(r'<think>.*?</think>', '', s, flags=re.I | re.S).strip()

    start = s.find('{')
    if start < 0:
        raise ValueError(f'no JSON object start found: {s[:300]}')

    depth = 0
    in_str = False
    esc = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = s[start:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        candidate = re.sub(r',\s*([}\]])', r'\1', candidate)
                        try:
                            return json.loads(candidate)
                        except Exception:
                            return ast.literal_eval(candidate)
    raise ValueError(f'no balanced JSON object found: {s[:700]}')


def clean_str(x: Any, max_chars: int = 240) -> str:
    s = re.sub(r'\s+', ' ', str(x or '')).strip()
    return s[:max_chars].rstrip()


def clean_list(x: Any, max_items: int, max_chars: int = 80) -> list[str]:
    if x is None:
        return []
    if isinstance(x, str):
        x = [x]
    if not isinstance(x, (list, tuple, set)):
        return []
    out, seen = [], set()
    for item in x:
        s = clean_str(item, max_chars=max_chars)
        if not s:
            continue
        key = s.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
        if len(out) >= max_items:
            break
    return out


def normalize_descriptor(obj: dict[str, Any]) -> dict[str, Any]:
    d = {}
    d['legal_area'] = clean_str(obj.get('legal_area'), 80)
    d['primary_domain'] = clean_str(obj.get('primary_domain'), 80)
    d['secondary_domain'] = clean_str(obj.get('secondary_domain'), 100)
    d['legal_domain_path'] = clean_list(obj.get('legal_domain_path'), 5, 60)
    d['topic'] = clean_str(obj.get('topic'), 100)
    d['subtopic'] = clean_str(obj.get('subtopic'), 120)
    d['micro_topic'] = clean_str(obj.get('micro_topic'), 160)
    d['concepts_en'] = clean_list(obj.get('concepts_en'), 6, 70)
    d['terms_original'] = clean_list(obj.get('terms_original'), 8, 100)
    d['doctrinal_rule'] = clean_str(obj.get('doctrinal_rule'), 260)
    d['legal_test'] = clean_str(obj.get('legal_test'), 220)
    d['fact_pattern_tags'] = clean_list(obj.get('fact_pattern_tags'), 5, 70)
    d['procedural_context'] = clean_str(obj.get('procedural_context'), 120)

    role = clean_str(obj.get('paragraph_role'), 60).lower().replace(' ', '_').replace('-', '_')
    d['paragraph_role'] = role if role in ROLE_VALUES else 'neutral'
    d['authority_role'] = clean_list(obj.get('authority_role'), 3, 60)
    try:
        d['specificity_score'] = max(0.0, min(1.0, float(obj.get('specificity_score', 0))))
    except Exception:
        d['specificity_score'] = 0.0

    # Hard guarantee: forbidden/final fields never survive in raw descriptor.
    forbidden = {
        'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
        'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
        'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
    }
    for k in forbidden:
        d.pop(k, None)
    return d


def empty_descriptor(error: str = '') -> dict[str, Any]:
    return {
        'legal_area': '',
        'primary_domain': '',
        'secondary_domain': '',
        'legal_domain_path': [],
        'topic': '',
        'subtopic': '',
        'micro_topic': '',
        'concepts_en': [],
        'terms_original': [],
        'doctrinal_rule': '',
        'legal_test': '',
        'fact_pattern_tags': [],
        'procedural_context': '',
        'paragraph_role': 'neutral',
        'authority_role': [],
        'specificity_score': 0.0,
        '_descriptor_error': error[:500],
    }


In [7]:
# Cell 6 — Load vLLM with Qwen3-8B-AWQ

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

print('Loading tokenizer/model:', cfg.model_name)
tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_name,
    trust_remote_code=True,
    cache_dir=cfg.model_download_dir,
)

base_llm_kwargs = dict(
    model=cfg.model_name,
    trust_remote_code=True,
    tensor_parallel_size=cfg.tensor_parallel_size,
    gpu_memory_utilization=cfg.gpu_memory_utilization,
    max_model_len=cfg.max_model_len,
    max_num_seqs=cfg.max_num_seqs,
    enforce_eager=cfg.enforce_eager,
    disable_custom_all_reduce=cfg.disable_custom_all_reduce,
    disable_log_stats=True,
    download_dir=cfg.model_download_dir,
)

# vLLM quantization names can differ across versions/builds. Try the requested AWQ mode first,
# then AWQ Marlin, then auto-detection as a final fallback.
quantization_candidates = []
for q in [cfg.quantization, 'awq_marlin', None]:
    if q not in quantization_candidates:
        quantization_candidates.append(q)

last_error = None
for quantization in quantization_candidates:
    llm_kwargs = dict(base_llm_kwargs)
    if quantization is not None:
        llm_kwargs['quantization'] = quantization
    try:
        print(f'Initializing vLLM with quantization={quantization!r}')
        llm = LLM(**llm_kwargs)
        print('vLLM loaded')
        break
    except Exception as exc:
        last_error = exc
        print(f'vLLM load failed with quantization={quantization!r}: {repr(exc)}')
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    raise RuntimeError('Could not load Qwen3-8B-AWQ with vLLM') from last_error


Loading tokenizer/model: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1
[vLLM] using explicit TRITON attention_config
INFO 05-03 07:32:10 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.78, 'max_num_seqs': 8, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'enforce_eager': True, 'disable_custom_all_reduce': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, tq_max_kv_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False, use_fp4_indexer_cache=False), 'model': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1'}
WARNING 

[W503 07:32:48.562628145 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


(EngineCore pid=446) INFO 05-03 07:32:49 [gpu_model_runner.py:4777] Starting to load model /kaggle/input/models/qwen-lm/qwen-3/transformers/8b-awq/1...
(EngineCore pid=446) INFO 05-03 07:32:49 [awq_marlin.py:420] Using MarlinLinearKernel for AWQMarlinLinearMethod
(EngineCore pid=446) INFO 05-03 07:32:49 [cuda.py:308] Using AttentionBackendEnum.TRITON_ATTN backend.
(EngineCore pid=446) INFO 05-03 07:32:49 [weight_utils.py:904] Filesystem type for checkpoints: NFS. Checkpoint size: 5.68 GiB. Available RAM: 19.38 GiB.
(EngineCore pid=446) INFO 05-03 07:32:49 [weight_utils.py:874] Prefetching checkpoint files into page cache started (in background)


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=446) INFO 05-03 07:33:00 [weight_utils.py:851] Prefetching checkpoint files: 10% (1/2)


Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:53<00:53, 53.50s/it]


(EngineCore pid=446) INFO 05-03 07:33:43 [weight_utils.py:851] Prefetching checkpoint files: 20% (2/2)
(EngineCore pid=446) INFO 05-03 07:33:43 [weight_utils.py:869] Prefetching checkpoint files into page cache finished in 54.15s


Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:55<00:00, 22.92s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:55<00:00, 27.50s/it]
(EngineCore pid=446) 


(EngineCore pid=446) INFO 05-03 07:33:44 [default_loader.py:384] Loading weights took 55.10 seconds
(EngineCore pid=446) INFO 05-03 07:33:47 [gpu_model_runner.py:4879] Model loading took 5.71 GiB memory and 57.270029 seconds
(EngineCore pid=446) INFO 05-03 07:34:07 [gpu_worker.py:440] Available KV cache memory: 4.79 GiB
(EngineCore pid=446) INFO 05-03 07:34:07 [kv_cache_utils.py:1711] GPU KV cache size: 34,848 tokens
(EngineCore pid=446) INFO 05-03 07:34:07 [kv_cache_utils.py:1716] Maximum concurrency for 4,096 tokens per request: 8.51x
(EngineCore pid=446) INFO 05-03 07:34:07 [core.py:306] init engine (profile, create kv cache, warmup model) took 20.42 s
(EngineCore pid=446) INFO 05-03 07:34:08 [vllm.py:840] Asynchronous scheduling is enabled.
(EngineCore pid=446) WARNING 05-03 07:34:08 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=446) WARNING 05-03 07:34:08 [vllm.py:914] In

In [8]:
# Cell 7 — Generation helpers

def render_prompt(citation: str, text: str, repair: bool = False, bad_output: str = '', error: str = '') -> str:
    user_prompt = build_user_prompt(citation, text)
    if repair:
        user_prompt = f'''The previous output was invalid JSON.

Parser error:
{error}

Previous output:
{bad_output[:1600]}

Repair by returning exactly one complete compact JSON object using the same schema.
Do not add questions, summaries, anchors, outcomes, or retrieval views.

{user_prompt}'''

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=cfg.enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_raw(prompts: list[str], max_tokens: int) -> list[str]:
    params = SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=max_tokens,
        repetition_penalty=cfg.repetition_penalty,
    )
    outputs = llm.generate(prompts, sampling_params=params, use_tqdm=False)
    return [out.outputs[0].text if out.outputs else '' for out in outputs]


def generate_descriptor(citation: str, text: str, first_raw: str | None = None) -> tuple[dict[str, Any], dict[str, Any]]:
    attempts = []
    raw = first_raw
    for attempt in range(cfg.max_retries + 1):
        try:
            if raw is None:
                raw = generate_raw([render_prompt(citation, text)], cfg.max_new_tokens)[0]
            obj = extract_json_object(raw)
            desc = normalize_descriptor(obj)
            return desc, {
                'status': 'ok' if attempt == 0 else 'ok_after_retry',
                'attempt_count': attempt + 1,
                'error': None,
                'raw_output': raw if cfg.include_raw_output_on_success else None,
            }
        except Exception as exc:
            err = repr(exc)
            attempts.append({'attempt': attempt + 1, 'error': err, 'raw_output': (raw or '')[:1600]})
            if attempt >= cfg.max_retries:
                return empty_descriptor(err), {
                    'status': 'failed_descriptor_parse',
                    'attempt_count': attempt + 1,
                    'error': err,
                    'attempts': attempts,
                    'raw_output': raw,
                }
            raw = generate_raw([render_prompt(citation, text, repair=True, bad_output=raw or '', error=err)], cfg.retry_max_new_tokens)[0]


In [9]:
# Cell 8 — Run descriptor extraction and write raw LLM descriptor JSONL

records = []
failures = []
t0 = time.time()

for start in tqdm(range(0, len(work_df), cfg.batch_size), desc='llm-descriptor batches'):
    batch = work_df.iloc[start:start + cfg.batch_size]
    row_objs = []
    prompts = []
    for _, row in batch.iterrows():
        citation = str(row[citation_col])
        text = str(row[text_col])
        row_obj = {
            '_source_row': int(row['_source_row']),
            'citation': citation,
            'text': text,
        }
        row_objs.append(row_obj)
        prompts.append(render_prompt(citation, text))

    try:
        raws = generate_raw(prompts, cfg.max_new_tokens)
    except Exception as exc:
        print('Batch generation failed; falling back to single-row generation:', repr(exc))
        raws = [None] * len(row_objs)

    for row_obj, raw in zip(row_objs, raws):
        desc, gen = generate_descriptor(row_obj['citation'], row_obj['text'], first_raw=raw)
        rec = {
            '_source_row': row_obj['_source_row'],
            'citation': row_obj['citation'],
            'text': row_obj['text'],
            'llm_enrichment': desc,
            'llm_generation': {
                'model': cfg.model_name,
                'method': 'minimal_descriptor_only',
                **gen,
            },
        }
        records.append(rec)
        if gen['status'].startswith('failed'):
            failures.append(rec)

elapsed = time.time() - t0

with output_jsonl.open('w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

if failures:
    with output_failures_jsonl.open('w', encoding='utf-8') as f:
        for rec in failures:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
else:
    if output_failures_jsonl.exists():
        output_failures_jsonl.unlink()

preview_rows = []
for rec in records:
    e = rec['llm_enrichment']
    g = rec['llm_generation']
    preview_rows.append({
        '_source_row': rec['_source_row'],
        'citation': rec['citation'],
        'status': g['status'],
        'legal_area': e.get('legal_area'),
        'primary_domain': e.get('primary_domain'),
        'secondary_domain': e.get('secondary_domain'),
        'topic': e.get('topic'),
        'subtopic': e.get('subtopic'),
        'micro_topic': e.get('micro_topic'),
        'concepts_en': ' | '.join(e.get('concepts_en', [])),
        'terms_original': ' | '.join(e.get('terms_original', [])),
        'doctrinal_rule': e.get('doctrinal_rule'),
        'legal_test': e.get('legal_test'),
        'fact_pattern_tags': ' | '.join(e.get('fact_pattern_tags', [])),
        'procedural_context': e.get('procedural_context'),
        'paragraph_role': e.get('paragraph_role'),
        'authority_role': ' | '.join(e.get('authority_role', [])),
        'specificity_score': e.get('specificity_score'),
    })
preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(output_preview_csv, index=False)

metrics = {
    'start': cfg.start,
    'limit': cfg.limit,
    'selected_rows': len(work_df),
    'written_rows': len(records),
    'failures': len(failures),
    'elapsed_seconds': elapsed,
    'rows_per_second': len(records) / max(elapsed, 1e-9),
    'status_counts': dict(Counter(rec['llm_generation']['status'] for rec in records)),
    'output_jsonl': str(output_jsonl),
    'output_preview_csv': str(output_preview_csv),
    'output_failures_jsonl': str(output_failures_jsonl) if failures else None,
}
output_metrics_json.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(metrics, indent=2))
display(preview_df)


llm-descriptor batches:   0%|          | 0/13 [00:00<?, ?it/s]

{
  "start": 0,
  "limit": 50,
  "selected_rows": 50,
  "written_rows": 50,
  "failures": 0,
  "elapsed_seconds": 211.5400824546814,
  "rows_per_second": 0.23636182523806848,
  "status_counts": {
    "ok": 50
  },
  "output_jsonl": "/kaggle/working/court_llm_descriptors_0000000_0000049.jsonl",
  "output_preview_csv": "/kaggle/working/court_llm_descriptors_0000000_0000049_preview.csv",
  "output_failures_jsonl": null
}


,_source_row,citation,status,legal_area,primary_domain,secondary_domain,topic,subtopic,micro_topic,concepts_en,terms_original,doctrinal_rule,legal_test,fact_pattern_tags,procedural_context,paragraph_role,authority_role,specificity_score
0,1,BGE 139 I 2 E. 2,ok,Administrative law,Public law,Administrative procedure,Administrative review,Remand for reconsideration,Remand for re-examination and re-evaluation,Administrative review | Remand | Reconsiderati...,Rückweisung | Neubehandlung | Sachverhaltsabkl...,,,Administrative appeal | Reconsideration | Re-e...,Administrative appeal,reasoning,application_of_rule | legal_standard,0.8
1,2,BGE 139 I 2 E. 5.1,ok,Constitutional law,Public law,Local government law,Constitutional compliance,Compatibility of local decisions with referendums,Consistency of municipal zoning decisions with...,Constitutional review | Municipal authority | ...,Volksentscheid | Umzonung | Gemeinderat | Geme...,,Whether Art. 34 BV is respected,Municipal decision | Referendum initiative | L...,Constitutional review of municipal decision,reasoning,legal_standard | application_of_rule,0.8
2,3,BGE 139 I 2 E. 5.2,ok,Constitutional law,Public law,Federalism and local governance,Political rights protection,Initiative rights in local matters,Protection of initiative rights in municipal a...,Federalism | Political rights | Initiative rig...,Art. 34 Abs. 1 BV | Initiativrecht | Gemeinden...,,,,,reasoning,background,0.7
3,4,BGE 139 I 2 E. 5.3,ok,Constitutional law,Public law,Administrative law,Initiative validity,Procedural validity of initiatives,Validity of initiative proceedings before admi...,Initiative | Administrative law | Procedural v...,Initiative | Abstimmung | Verwaltungsgericht |...,,,Completed proceedings | Administrative court |...,Prior judicial review,procedural_history,background,0.3
4,5,BGE 139 I 2 E. 7.1,ok,Administrative law,Public law,Local government law,Initiative validity,Constitutional compatibility of initiatives,Scope of municipal authority in assessing init...,Initiative validity | Constitutional review | ...,Initiative | Vereinbarkeit mit höherrangigem R...,,,Municipal decision | Constitutional review | I...,Municipal council decision,reasoning,background | application_of_rule,0.8
5,6,BGE 139 I 2 E. 5.4,ok,Administrative Law,Public Law,Planning and Development,Planning Implementation,Dispute over planning initiative execution,Dispute over implementation of planning initia...,planning initiative | dispute resolution | adm...,Planungsinitiative | Verwaltungsgericht | Ersc...,,,dispute over implementation | planning initiative,Appeal to administrative court,facts,background,0.3
6,7,BGE 139 I 2 E. 7.1,ok,Administrative law,Public administration,Planning and land use,Administrative procedure,Right of objection and appeal,Right of objection in planning procedures,Administrative procedure | Planning law | Righ...,Nutzungsplanerlassverfahren | Einsprachebefugn...,,,Planning initiative | Administrative appeal | ...,Administrative appeal process,reasoning,application_of_rule | legal_standard,0.8
7,8,BGE 139 I 2 E. 5.5,ok,Administrative law,Public law,Local government law,Municipal decision-making,Amendment of zoning plans,Validity of amendments to zoning plans at muni...,Municipal authority | Zoning plan amendment | ...,Gemeindeversammlung | Urnenabstimmung | Zonen-...,,,Municipal assembly | Zoning plan | Amendment |...,Municipal decision review,reasoning,background | legal_standard | application_of_rule,0.8
8,9,BGE 139 I 2 E. 5.6,ok,Constitutional Law,Public Law,Initiative and Referendum,Initiative Implementation,Constitutional Interpretation and Implementation,Comparative analysis of initiative implementat...,constitutional interpretation | initiative imp...,Initiative | Verfassungsinitiative | Gesetzesi...,,,unformulated initiative | constitutional compl...,Constitutional interpretation,reasoning,constitutional interpretation | application_of...,0.8
9,10,BGE 139 I 2 E. 5.7,ok,Constitutional law,Public la

In [10]:
# Cell 9 — QC: verify this is raw descriptor output only

FORBIDDEN = {
    'statute_anchors', 'case_anchors', 'normalized_anchors', 'retrieval_views',
    'outcome_signal', 'query_phrases_en', 'natural_language_queries', 'legal_question',
    'summary_en', 'english_summary', 'enrichment_quality', 'anchor_quality_flags',
}

def find_forbidden(obj: Any, path: str = '') -> list[str]:
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f'{path}.{k}' if path else k
            if k in FORBIDDEN:
                hits.append(p)
            hits.extend(find_forbidden(v, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(find_forbidden(v, f'{path}[{i}]'))
    return hits

qc = []
for rec in records:
    e = rec['llm_enrichment']
    qc.append({
        'citation': rec['citation'],
        'status': rec['llm_generation']['status'],
        'forbidden_fields': find_forbidden(rec),
        'concept_count': len(e.get('concepts_en', [])),
        'terms_original_count': len(e.get('terms_original', [])),
        'has_topic': bool(e.get('topic') or e.get('subtopic') or e.get('micro_topic')),
        'specificity_score': e.get('specificity_score'),
    })
qc_df = pd.DataFrame(qc)
display(qc_df)
print('Forbidden field rows:', int(qc_df['forbidden_fields'].apply(bool).sum()))
print('Failed rows:', int(qc_df['status'].str.startswith('failed').sum()))


,citation,status,forbidden_fields,concept_count,terms_original_count,has_topic,specificity_score
0,BGE 139 I 2 E. 2,ok,[],4,4,True,0.8
1,BGE 139 I 2 E. 5.1,ok,[],4,6,True,0.8
2,BGE 139 I 2 E. 5.2,ok,[],4,4,True,0.7
3,BGE 139 I 2 E. 5.3,ok,[],4,4,True,0.3
4,BGE 139 I 2 E. 7.1,ok,[],4,6,True,0.8
5,BGE 139 I 2 E. 5.4,ok,[],3,3,True,0.3
6,BGE 139 I 2 E. 7.1,ok,[],4,6,True,0.8
7,BGE 139 I 2 E. 5.5,ok,[],5,6,True,0.8
8,BGE 139 I 2 E. 5.6,ok,[],5,7,True,0.8
9,BGE 139 I 2 E. 5.7,ok,[],3,4,True,0.3


Forbidden field rows: 0
Failed rows: 0


## Next local step

The notebook writes outputs under:

```text
/content/drive/MyDrive/swiss_law/outputs/
```

Use the generated `court_llm_descriptors_*.jsonl` as input to the local finalizer, which should call:

```python
normalize_enriched_court_row(
    citation=citation,
    text=text,
    llm_enrichment=raw["llm_enrichment"],
    deterministic_metadata=metadata,
)
```

That local step creates the final production JSONL with:

```text
rag_enrichment
normalized_anchors
anchor_quality_flags
retrieval_views
enrichment_quality
```
